# 05 - Final Validation

## Purpose

Run final validation across the completed capstone.

This notebook confirms that the key tables and views still exist, return rows, and match their expected reporting grains.

## Main checks

- important Silver tables exist
- important Gold tables exist
- Night Shift tables exist
- important views exist
- row counts are greater than zero
- key dimensions are not null
- duplicate grains are checked
- dashboard-ready outputs return rows

## Main idea

Final validation proves that the project is ready to present after Day 5 hardening.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog = "vattenfall_dev"

final_tables = [
    # Bronze
    f"{catalog}.raw.bronze_market_prices",
    f"{catalog}.raw.bronze_weather",
    f"{catalog}.raw.bronze_grid_events",
    f"{catalog}.raw.bronze_asset_reference",

    # Silver
    f"{catalog}.refined.silver_market_prices",
    f"{catalog}.refined.silver_weather",
    f"{catalog}.refined.silver_grid_events",
    f"{catalog}.refined.silver_asset_reference",
    f"{catalog}.refined.silver_regional_operations_base",

    # Gold
    f"{catalog}.analytics.gold_daily_market_summary",
    f"{catalog}.analytics.gold_weather_impact_summary",
    f"{catalog}.analytics.gold_grid_incident_summary",
    f"{catalog}.analytics.gold_regional_operations_dashboard",

    # Night Shift
    f"{catalog}.analytics.gold_data_trust_audit",
    f"{catalog}.analytics.gold_asset_incident_intelligence",
    f"{catalog}.analytics.gold_weather_grid_risk_correlation",
    f"{catalog}.analytics.gold_market_operations_stress",
    f"{catalog}.analytics.gold_pipeline_observability_summary",
    f"{catalog}.analytics.gold_executive_energy_risk_dashboard_base",
]

final_views = [
    f"{catalog}.analytics.vw_regional_operations_dashboard",
    f"{catalog}.analytics.vw_daily_incident_trends",
    f"{catalog}.analytics.vw_market_weather_overview",
    f"{catalog}.analytics.vw_executive_energy_risk_dashboard",
]

In [0]:
validation_rows = []

for object_name in final_tables + final_views:
    df = spark.table(object_name)

    validation_rows.append(
        Row(
            object_name=object_name,
            row_count=df.count(),
            column_count=len(df.columns),
            validation_status="PASS" if df.count() > 0 else "FAIL",
        )
    )

final_validation_df = spark.createDataFrame(validation_rows)

display(final_validation_df)

In [0]:
empty_outputs = final_validation_df.filter(F.col("row_count") <= 0)

if empty_outputs.count() > 0:
    display(empty_outputs)
    raise ValueError("Final validation failed: one or more tables/views are empty.")

print("Final row count validation passed.")

In [0]:
null_checks = {
    f"{catalog}.refined.silver_market_prices": ["report_day", "region"],
    f"{catalog}.refined.silver_weather": ["report_day", "region"],
    f"{catalog}.refined.silver_grid_events": ["event_day", "region", "asset_id"],
    f"{catalog}.analytics.gold_daily_market_summary": ["report_day", "region"],
    f"{catalog}.analytics.gold_weather_impact_summary": ["report_day", "region"],
    f"{catalog}.analytics.gold_grid_incident_summary": ["event_day", "region", "severity_band"],
    f"{catalog}.analytics.gold_regional_operations_dashboard": ["report_day", "region"],
    f"{catalog}.analytics.vw_executive_energy_risk_dashboard": ["report_day", "region", "executive_risk_status"],
}

null_rows = []

for object_name, columns in null_checks.items():
    df = spark.table(object_name)

    for column_name in columns:
        null_rows.append(
            Row(
                object_name=object_name,
                column_name=column_name,
                null_count=df.filter(F.col(column_name).isNull()).count(),
            )
        )

null_check_df = spark.createDataFrame(null_rows)

display(null_check_df)

In [0]:
grain_checks = {
    f"{catalog}.analytics.gold_daily_market_summary": ["report_day", "region"],
    f"{catalog}.analytics.gold_weather_impact_summary": ["report_day", "region"],
    f"{catalog}.analytics.gold_grid_incident_summary": ["event_day", "region", "severity_band"],
    f"{catalog}.analytics.gold_regional_operations_dashboard": ["report_day", "region"],
    f"{catalog}.analytics.gold_asset_incident_intelligence": ["asset_id"],
    f"{catalog}.analytics.gold_weather_grid_risk_correlation": ["report_day", "region"],
    f"{catalog}.analytics.gold_market_operations_stress": ["report_day", "region"],
    f"{catalog}.analytics.vw_executive_energy_risk_dashboard": ["report_day", "region"],
}

duplicate_rows = []

for object_name, grain_columns in grain_checks.items():
    df = spark.table(object_name)

    duplicate_count = (
        df
        .groupBy(*grain_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    duplicate_rows.append(
        Row(
            object_name=object_name,
            grain=", ".join(grain_columns),
            duplicate_count=duplicate_count,
        )
    )

duplicate_check_df = spark.createDataFrame(duplicate_rows)

display(duplicate_check_df)

In [0]:
market_df = spark.table(f"{catalog}.analytics.gold_daily_market_summary")
weather_df = spark.table(f"{catalog}.analytics.gold_weather_impact_summary")
grid_df = spark.table(f"{catalog}.analytics.gold_grid_incident_summary")
dashboard_df = spark.table(f"{catalog}.analytics.gold_regional_operations_dashboard")

sanity_rows = [
    Row(
        check_name="negative_market_volume",
        invalid_count=market_df.filter(F.col("total_volume_mwh") < 0).count(),
    ),
    Row(
        check_name="negative_wind_speed",
        invalid_count=weather_df.filter(F.col("avg_wind_speed_kmh") < 0).count(),
    ),
    Row(
        check_name="negative_incident_duration",
        invalid_count=grid_df.filter(F.col("total_duration_minutes") < 0).count(),
    ),
    Row(
        check_name="negative_dashboard_incident_count",
        invalid_count=dashboard_df.filter(F.col("total_incident_count") < 0).count(),
    ),
]

sanity_check_df = spark.createDataFrame(sanity_rows)

display(sanity_check_df)

In [0]:
dashboard_view = f"{catalog}.analytics.vw_regional_operations_dashboard"
executive_view = f"{catalog}.analytics.vw_executive_energy_risk_dashboard"

print("Regional operations dashboard view:")
display(spark.table(dashboard_view).limit(20))

print("Executive energy risk dashboard view:")
display(spark.table(executive_view).limit(20))

In [0]:
print("Final project validation completed.")
print("Bronze, Silver, Gold, Night Shift outputs, and reporting views are available and populated.")
print("Key null checks, duplicate grain checks, and metric sanity checks are visible.")
print("The project is ready for presentation.")